# 1. 사용 라이브러리/하이퍼 파라미터 정의

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.preprocessing import OneHotEncoder


target_path = None
dataset_path = None

target_col = 'is_canceled'

missing_ratio_threshold = 50.0

outlier_summary = {}


# 2. Kaggle 데이터 셋 다운로드 및 로드
## KaggleHub를 사용하면 공개된 Kaggle 데이터셋은 별도의 로그인 과정 없이 코드에서 바로 다운로드할 수 있습니다. 접근 권한이 필요한 데이터셋은 Kaggle 인증이 필요할 수 있습니다.

In [28]:
path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

for file_path in os.listdir(path):
  if file_path.endswith('.csv'):
    target_path = file_path
    break

dataset_path = os.path.join(path, target_path)

df = pd.read_csv(dataset_path)

print('downloaded dataset path: {}'.format(path))
print('csv file path: {}'.format(dataset_path))
print('dataset shape: {}'.format(df.shape))

downloaded dataset path: C:\Users\rkd76\.cache\kagglehub\datasets\jessemostipak\hotel-booking-demand\versions\1
csv file path: C:\Users\rkd76\.cache\kagglehub\datasets\jessemostipak\hotel-booking-demand\versions\1\hotel_bookings.csv
dataset shape: (119390, 32)


In [29]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


# 3. Explatory Data Analysis (EDA)

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [31]:
df.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,103050.000000,6797.000000,119390.000000,119390.000000,119390.000000,119390.000000
mean,0.370416,104.011416,2016.156554,27.165173,15.798241,0.927599,2.500302,1.856403,0.103890,0.007949,0.031912,0.087118,0.137097,0.221124,86.693382,189.266735,2.321149,101.831122,0.062518,0.571363
std,0.482918,106.863097,0.707476,13.605138,8.780829,0.998613,1.908286,0.579261,0.398561,0.097436,0.175767,0.844336,1.497437,0.652306,110.774548,131.655015,17.594721,50.535790,0.245291,0.792798
min,0.000000,0.000000,2015.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,0.000000,-6.380000,0.000000,0.000000
25%,0.000000,18.000000,2016.000000,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,62.000000,0.000000,69.290000,0.000000,0.000000
50%,0.000000,69.000000,2016.000000,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.000000,179.000000,0.000000,94.575000,0.000000,0.000000
75%,1.000000,160.000000,2017.000000,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,229.000000,270.000000,0.000000,126.000000,0.000000,1.000000
max,1.000000,737.000000,2017.000000,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,1.000000,26.000000,72.000000,21.000000,535.000000,543.000000,391.000000,5400.000000,8.000000,5.000000


In [32]:
df = df.drop(columns=['reservation_status', 'reservation_status_date'])

label_info = df[target_col].value_counts()
label_dict = label_info.to_dict()

print('data 정보 확인: {}'.format(df.shape))

print('label 정보 확인: {}'.format(label_dict))
print('label 분포 확인: {}'.format(label_info / (label_dict[0] + label_dict[1])))

data 정보 확인: (119390, 30)
label 정보 확인: {0: 75166, 1: 44224}
label 분포 확인: is_canceled
0    0.629584
1    0.370416
Name: count, dtype: float64


# 4. 중복 데이터 처리 (제거)
## 서로 다른 데이터 (유의미한 데이터)가 동일 값을 갖는 경우가 있으므로 모든 중복 데이터를 제거하는 것은 아니나, 이번 실습에서는 동일 데이터는 중복 데이터를 가정하고 제거합니다

In [33]:
n_duplicates = df.duplicated().sum()

before = df.shape[0]
df = df.drop_duplicates(keep='first').reset_index(drop=True)
after = df.shape[0]

print("중복된 행의 개수: {}".format(n_duplicates))
print("제거 전 행 개수: {}".format(before))
print("제거 후 행 개수: {}".format(after))
print("제거된 중복 행 개수: {}".format(before-after))

중복된 행의 개수: 32252
제거 전 행 개수: 119390
제거 후 행 개수: 87138
제거된 중복 행 개수: 32252


# 5. 결측치 처리

In [34]:
num_missing = df.isnull().sum()
missing_ratio = (num_missing / len(df)) * 100

print('결측치 개수:\n{}\n'.format(num_missing[num_missing > 0]))
print('결측치 비율:\n{}'.format(missing_ratio[missing_ratio > 0]))

결측치 개수:
children        4
country       451
agent       12160
company     81890
dtype: int64

결측치 비율:
children     0.004590
country      0.517570
agent       13.954876
company     93.977369
dtype: float64


In [35]:
# 삭제를 고려할 컬럼 후보 출력
del_cand_cols = missing_ratio[missing_ratio > missing_ratio_threshold].index.tolist()

for col in del_cand_cols:
    print("{}: 결측  비율: {}%".format(col, missing_ratio[col]))

user_input = input("삭제할 칼럼명을 ','로 구분해서 입력해주세요\n")

company: 결측  비율: 93.97736923041612%


In [36]:
del_cols = [c.strip() for c in user_input.split(',') if c.strip() != '']
print("삭제할 컬럼:", del_cols)

# 입력받은 컬럼 삭제
if del_cols:
    df = df.drop(columns=del_cols, errors='ignore') #errors='ignore' 은 내가 따로 추가함
    print("{} 컬럼을 삭제했습니다.".format(del_cols))
else:
    print("삭제한 컬럼이 없습니다.")

print("\nshape:", df.shape)

# 나머지 결측치는 수치형은 중앙값, 범주형은 최빈값으로 대체
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            print("{} -> 중앙값 {}으로 대체".format(col, fill_value))
        else:
            fill_value = df[col].mode()[0]
            df[col] = df[col].fillna(fill_value)
            print("{} -> 최빈값 {}으로 대체".format(col, fill_value))

print("\n남은 결측치 개수:", df.isnull().sum().sum())

삭제할 컬럼: []
삭제한 컬럼이 없습니다.

shape: (87138, 30)
children -> 중앙값 0.0으로 대체
country -> 최빈값 PRT으로 대체
agent -> 중앙값 14.0으로 대체
company -> 중앙값 169.0으로 대체

남은 결측치 개수: 0


# 6. Outlier 처리 - Inter Quartile range (IQR) 기반 Outlier 제거

In [37]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]
# numeric_cols = ['stays_in_weekend_nights', 'stays_in_week_nights']
print("수치형 컬럼(타겟 제외):", numeric_cols)

def get_iqr_bounds(df_col):
    q1 = df_col.quantile(0.25)
    q3 = df_col.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return lower, upper

for numeric_col in numeric_cols:
    lower, upper = get_iqr_bounds(df[numeric_col])
    num_outliers = ((df[numeric_col] < lower) | (df[numeric_col] > upper)).sum()
    outlier_summary[numeric_col] = {'하한': lower, '상한': upper, '이상치 개수': num_outliers}


before = df.shape[0]
for col in numeric_cols:
    lower, upper = get_iqr_bounds(df[col])
    df = df[(df[col] >= lower) & (df[col] <= upper)]
df = df.reset_index(drop=True)
after = df.shape[0]

print('outlier 정보: {}'.format(outlier_summary))
print("데이터 개수: {} -> {}".format(before, after))

수치형 컬럼(타겟 제외): ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
outlier 정보: {'lead_time': {'하한': np.float64(-160.0), '상한': np.float64(296.0), '이상치 개수': np.int64(2371)}, 'arrival_date_year': {'하한': np.float64(2014.5), '상한': np.float64(2018.5), '이상치 개수': np.int64(0)}, 'arrival_date_week_number': {'하한': np.float64(-15.5), '상한': np.float64(68.5), '이상치 개수': np.int64(0)}, 'arrival_date_day_of_month': {'하한': np.float64(-14.5), '상한': np.float64(45.5), '이상치 개수': np.int64(0)}, 'stays_in_weekend_nights': {'하한': np.float64(-3.0), '상한': np.float64(5.0), '이상치 개수': np.int64(218)}, 'stays_in_week_nights': {'하한': np.float64(-3.5), '상한': np.float64(8.5), '이상치 개수': np.int64

# 7. Numeric Data 처리 - Data Scaling (Standardization / min-max normalization)

In [38]:
df_standardized = df.copy()

for numeric_col in numeric_cols:
    mean = df[numeric_col].mean()
    std = df[numeric_col].std()
    df_standardized[numeric_col] = (df[numeric_col] - mean) / std

df_standardized[numeric_cols].describe().loc[['mean', 'std']]

,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
mean,-4.204617e-17,-1.248171e-13,7.808574e-17,3.528875e-17,-6.607255e-17,-1.201319e-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,-3.123430e-16,NaN,-3.603957e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.000000e+00,NaN,1.000000e+00


In [39]:
df_normalized = df.copy()

for numeric_col in numeric_cols:
    min_val = df[numeric_col].min()
    max_val = df[numeric_col].max()
    df_normalized[numeric_col] = (df[numeric_col] - min_val) / (max_val - min_val)

df_normalized[numeric_cols].describe().loc[['min', 'max']]

,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
min,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0


# 8. Non-numeric data 처리 (One-hot Encoding)

In [40]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("범주형 컬럼:", categorical_cols)

for categorical_col in categorical_cols:
    print("{}: 고유값 개수 - {} / 고유 값 - {}".format(categorical_col, df[categorical_col].nunique(), df[categorical_col].unique()))

범주형 컬럼: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
hotel: 고유값 개수 - 2 / 고유 값 - <StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str
arrival_date_month: 고유값 개수 - 12 / 고유 값 - <StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str
meal: 고유값 개수 - 5 / 고유 값 - <StringArray>
['BB', 'FB', 'HB', 'Undefined', 'SC']
Length: 5, dtype: str
country: 고유값 개수 - 158 / 고유 값 - <StringArray>
['GBR', 'PRT', 'IRL', 'ESP', 'ROU', 'NOR', 'USA', 'POL', 'BEL', 'DEU',
 ...
 'KEN', 'MRT', 'ABW', 'NCL', 'SDN', 'ATF', 'SLE', 'SLV', 'LAO', 'ETH']
Length: 158, dtype: str
market_segment: 고유값 개수 - 7 / 고유 값 - <StringArray>
[    'Online TA',        'Direct', 'Offline TA/TO',        'Groups',
     'Corporate', 'Complementary',     'Undefined']
Lengt

C:\Users\rkd76\AppData\Local\Temp\ipykernel_8608\2524852809.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


In [41]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

print("인코딩 전 컬럼 수:", df.shape[1])
print("인코딩 후 컬럼 수:", df_encoded.shape[1])

df_encoded.head()

인코딩 전 컬럼 수: 30
인코딩 후 컬럼 수: 235


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,assigned_room_type_H,assigned_room_type_I,assigned_room_type_K,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient-Party
0,0,14,2015,27,1,0,2,2,0.0,0,...,False,False,False,True,False,False,False,False,True,False
1,0,0,2015,27,1,0,2,2,0.0,0,...,False,False,False,True,False,False,False,False,True,False
2,0,9,2015,27,1,0,2,2,0.0,0,...,False,False,False,True,False,False,False,False,True,False
3,1,85,2015,27,1,0,3,2,0.0,0,...,False,False,False,True,False,False,False,False,True,False
4,1,75,2015,27,1,0,3,2,0.0,0,...,False,False,False,True,False,False,False,False,True,False
